# Semana 6 — OMOP CDM v5.4: ETL de Synthea al modelo común de OHDSI

**Objetivo:** transformar silver al **OMOP Common Data Model v5.4** (tablas `person`, `observation_period`, `visit_occurrence`, `condition_occurrence`, `measurement`, `death`) siguiendo las convenciones de mapeo que publica OHDSI en su ETL oficial `ETL-Synthea`, y evaluar el resultado con chequeos al estilo del *Data Quality Dashboard*.

**Productos reales usados:** OMOP CDM v5.4 (OHDSI), DuckDB. Opcional: vocabularios estándar descargados de Athena (athena.ohdsi.org, cuenta gratuita).

**Al terminar tendrás:** `data/omop/*.parquet` y `data/reports/omop_dq.csv`.

**Requisitos:** semanas 3 y 5.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — DuckDB con silver cargado

**Explicación línea por línea**

- `%pip install -q "duckdb>=1.1"` e `import duckdb`: igual que la semana 3.
- `con = duckdb.connect()` y `SET TimeZone = 'UTC'`: base en memoria con zona fija.
- `CREATE SCHEMA ... silver; ... cdm; ... stg;`: tres espacios de nombres: `silver` (entrada), `cdm` (tablas OMOP; en las herramientas OHDSI este esquema se configura con el parámetro `cdmDatabaseSchema`) y `stg` (tablas intermedias de mapeo).
- `CREATE OR REPLACE VIEW silver.{t} AS SELECT * FROM read_parquet('...')`: expone las tablas de la semana 3 sin copiarlas.
- `OMOP_DIR`, `REPORTS_DIR`: carpetas de salida.

In [ ]:
%pip install -q "duckdb>=1.1"

In [ ]:
import duckdb
import pandas as pd

SILVER_DIR = DATA_DIR / "silver"
OMOP_DIR = DATA_DIR / "omop"
REPORTS_DIR = DATA_DIR / "reports"
for d in (OMOP_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
con.execute("CREATE SCHEMA IF NOT EXISTS silver; CREATE SCHEMA IF NOT EXISTS cdm; CREATE SCHEMA IF NOT EXISTS stg;")
for t in ["patients", "encounters", "conditions", "observations"]:
    con.execute(f"CREATE OR REPLACE VIEW silver.{t} AS SELECT * FROM read_parquet('{(SILVER_DIR / f'{t}.parquet').as_posix()}')")
con.execute("SELECT COUNT(*) FROM silver.patients").fetchall()[0]

### Paso 2 — Tablas de mapeo de identificadores (staging)

OMOP exige identificadores **enteros** (`person_id`, `visit_occurrence_id`), mientras Synthea usa UUID de texto. Se crean tablas de correspondencia en un esquema intermedio `stg`.

**Explicación línea por línea**

- `CREATE OR REPLACE TABLE stg.person_map AS SELECT ROW_NUMBER() OVER (ORDER BY patient_id) AS person_id, patient_id FROM silver.patients`: asigna 1, 2, 3... ordenando por UUID. **Por qué ordenar:** sin `ORDER BY`, el número asignado podría cambiar entre ejecuciones; con orden fijo el ETL es **determinista** y reproducible.
- `stg.visit_map`: igual para encuentros.
- `SELECT COUNT(*) ..., COUNT(DISTINCT ...)`: comprueba que el mapeo es 1 a 1.
- **En la práctica real** el UUID original se conserva en `*_source_value` para poder rastrear cualquier fila OMOP hasta su origen.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE stg.person_map AS
SELECT ROW_NUMBER() OVER (ORDER BY patient_id) AS person_id, patient_id
FROM silver.patients;

CREATE OR REPLACE TABLE stg.visit_map AS
SELECT ROW_NUMBER() OVER (ORDER BY encounter_id) AS visit_occurrence_id, encounter_id
FROM silver.encounters;
""")
con.sql("""
SELECT (SELECT COUNT(*) FROM stg.person_map) AS personas,
       (SELECT COUNT(DISTINCT patient_id) FROM stg.person_map) AS uuids_distintos,
       (SELECT COUNT(*) FROM stg.visit_map) AS visitas
""").df()

### Paso 3 — `cdm.person`

**Explicación línea por línea (SQL)**

- `CASE p.sex WHEN 'M' THEN 8507 WHEN 'F' THEN 8532 ELSE 0 END AS gender_concept_id`: 8507 (Male) y 8532 (Female) son los conceptos estándar del dominio *Gender*; 0 significa "sin concepto". Es la regla de `ETL-Synthea`. En CDM v5.4 este campo representa el **sexo biológico al nacer**, no la identidad de género.
- `EXTRACT(year FROM p.birth_date)` y similares: el año de nacimiento es obligatorio; mes y día son opcionales.
- `CAST(p.birth_date AS TIMESTAMP) AS birth_datetime`: fecha-hora a medianoche.
- `CASE lower(p.race) WHEN 'white' THEN 8527 WHEN 'black' THEN 8516 WHEN 'asian' THEN 8515 ELSE 0 END`: mapeo de raza usado por `ETL-Synthea`; el resto de valores queda en 0 y el texto original se conserva en `race_source_value`.
- `CASE lower(p.ethnicity) WHEN 'hispanic' THEN 38003563 WHEN 'nonhispanic' THEN 38003564 ELSE 0 END`: *Hispanic or Latino* / *Not Hispanic or Latino*.
- `p.patient_id AS person_source_value`: el UUID original, para trazabilidad.
- `JOIN stg.person_map m USING (patient_id)`: trae el entero asignado.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE cdm.person AS
SELECT
    m.person_id,
    CASE p.sex WHEN 'M' THEN 8507 WHEN 'F' THEN 8532 ELSE 0 END          AS gender_concept_id,
    EXTRACT(year FROM p.birth_date)::INTEGER                              AS year_of_birth,
    EXTRACT(month FROM p.birth_date)::INTEGER                             AS month_of_birth,
    EXTRACT(day FROM p.birth_date)::INTEGER                               AS day_of_birth,
    CAST(p.birth_date AS TIMESTAMP)                                       AS birth_datetime,
    CASE lower(p.race) WHEN 'white' THEN 8527 WHEN 'black' THEN 8516
                       WHEN 'asian' THEN 8515 ELSE 0 END                  AS race_concept_id,
    CASE lower(p.ethnicity) WHEN 'hispanic' THEN 38003563
                            WHEN 'nonhispanic' THEN 38003564 ELSE 0 END   AS ethnicity_concept_id,
    p.patient_id                                                          AS person_source_value,
    p.sex                                                                 AS gender_source_value,
    p.race                                                                AS race_source_value,
    p.ethnicity                                                           AS ethnicity_source_value
FROM silver.patients p
JOIN stg.person_map m USING (patient_id)
""")
con.sql("SELECT gender_concept_id, COUNT(*) AS n FROM cdm.person GROUP BY 1").df()

### Paso 4 — `cdm.observation_period` y `cdm.death`

En OMOP, un **periodo de observación** es el intervalo en que se asume que los eventos de la persona quedan registrados. Fuera de él, la ausencia de un diagnóstico no significa ausencia de enfermedad.

**Explicación línea por línea (SQL)**

- `MIN(e.start_ts)::DATE` / `MAX(e.stop_ts)::DATE`: primer y último encuentro de cada paciente; es la regla de `ETL-Synthea`.
- `32882 AS period_type_concept_id`: *Standard algorithm from EHR*, el tipo que usa `ETL-Synthea` para este periodo.
- `ROW_NUMBER() OVER (ORDER BY m.person_id) AS observation_period_id`: identificador entero determinista.
- `cdm.death`: una fila por persona fallecida; `32817` es el *type concept* "EHR" (origen del registro).
- `WHERE p.death_date IS NOT NULL`: solo fallecidos.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE cdm.observation_period AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id) AS observation_period_id,
       m.person_id,
       MIN(e.start_ts)::DATE AS observation_period_start_date,
       MAX(e.stop_ts)::DATE  AS observation_period_end_date,
       32882                 AS period_type_concept_id
FROM silver.encounters e
JOIN stg.person_map m USING (patient_id)
GROUP BY m.person_id;

CREATE OR REPLACE TABLE cdm.death AS
SELECT m.person_id, p.death_date, 32817 AS death_type_concept_id
FROM silver.patients p
JOIN stg.person_map m USING (patient_id)
WHERE p.death_date IS NOT NULL;
""")
con.sql("SELECT COUNT(*) AS periodos, (SELECT COUNT(*) FROM cdm.death) AS muertes FROM cdm.observation_period").df()

### Paso 5 — `cdm.visit_occurrence`

**Explicación línea por línea (SQL)**

- `CASE ... WHEN 'inpatient' THEN 9201 ... IN ('emergency', 'urgentcare') THEN 9203 ... IN ('ambulatory', 'wellness', 'outpatient') THEN 9202 ELSE 0 END`: 9201 *Inpatient Visit*, 9202 *Outpatient Visit*, 9203 *Emergency Room Visit*. Es exactamente la regla de `ETL-Synthea`.
- `32827 AS visit_type_concept_id`: *EHR encounter record*.
- `e.encounter_id AS visit_source_value`: trazabilidad.
- `e.encounter_class AS visit_source_concept_name`: **no** es una columna OMOP estándar; la añadimos para no perder la distinción `wellness`/`ambulatory`, que OMOP agrupa en 9202 y que usaremos en la semana 7. **Regla de práctica real:** documenta toda columna que no esté en la especificación oficial.
- `LAG(v.visit_occurrence_id) OVER (PARTITION BY m.person_id ORDER BY e.start_ts) AS preceding_visit_occurrence_id`: `LAG` trae la visita anterior de la misma persona; también es regla de `ETL-Synthea`.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE cdm.visit_occurrence AS
SELECT v.visit_occurrence_id,
       m.person_id,
       CASE WHEN e.encounter_class = 'inpatient' THEN 9201
            WHEN e.encounter_class IN ('emergency', 'urgentcare') THEN 9203
            WHEN e.encounter_class IN ('ambulatory', 'wellness', 'outpatient') THEN 9202
            ELSE 0 END                  AS visit_concept_id,
       e.start_ts::DATE                AS visit_start_date,
       e.start_ts                      AS visit_start_datetime,
       e.stop_ts::DATE                 AS visit_end_date,
       e.stop_ts                       AS visit_end_datetime,
       32827                           AS visit_type_concept_id,
       e.encounter_id                  AS visit_source_value,
       e.encounter_class               AS visit_source_concept_name,
       LAG(v.visit_occurrence_id) OVER (PARTITION BY m.person_id ORDER BY e.start_ts) AS preceding_visit_occurrence_id
FROM silver.encounters e
JOIN stg.person_map m USING (patient_id)
JOIN stg.visit_map v USING (encounter_id)
""")
con.sql("SELECT visit_concept_id, COUNT(*) AS n FROM cdm.visit_occurrence GROUP BY 1 ORDER BY 1").df()

### Paso 6 — Vocabularios estándar (opcional, con Athena) o concepto 0

En OMOP, los códigos de origen (SNOMED CT, LOINC...) se traducen a **conceptos estándar** con las tablas `CONCEPT` y `CONCEPT_RELATIONSHIP` (relación `Maps to`). Esas tablas se descargan gratis desde Athena con una cuenta, como archivos separados por tabulaciones.

**Explicación línea por línea**

- `VOCAB_DIR = DATA_DIR / "vocab"`: si subes ahí `CONCEPT.csv` y `CONCEPT_RELATIONSHIP.csv` de Athena, el notebook los usa.
- `HAS_VOCAB = (...).exists() and (...).exists()`: detecta si están.
- `read_csv('...', delim='\t', header=true, quote='', all_varchar=true)`: los archivos de Athena usan tabulador y no usan comillas; `all_varchar` evita que DuckDB adivine tipos en millones de filas.
- `stg.source_to_standard`: une el concepto de origen (`vocabulary_id` + `concept_code`) con su destino por `Maps to`, filtrando conceptos destino estándar (`standard_concept = 'S'`) y vigentes (`invalid_reason` vacío). Es el mismo patrón `CTE_TARGET_VOCAB_MAP` que documenta `ETL-Synthea`.
- Sin vocabularios: se crea una tabla vacía con las mismas columnas, y todos los `*_concept_id` quedarán en 0 con el código original en `*_source_value`. **Esto es válido en OMOP:** cuando un código de origen no puede traducirse a un concepto estándar, se guarda con concepto 0 y su valor de origen.

In [ ]:
VOCAB_DIR = DATA_DIR / "vocab"
HAS_VOCAB = (VOCAB_DIR / "CONCEPT.csv").exists() and (VOCAB_DIR / "CONCEPT_RELATIONSHIP.csv").exists()

if HAS_VOCAB:
    opts = "delim='\t', header=true, quote='', all_varchar=true"
    con.execute(f"CREATE OR REPLACE TABLE stg.concept AS SELECT * FROM read_csv('{(VOCAB_DIR / 'CONCEPT.csv').as_posix()}', {opts})")
    con.execute(f"CREATE OR REPLACE TABLE stg.concept_relationship AS SELECT * FROM read_csv('{(VOCAB_DIR / 'CONCEPT_RELATIONSHIP.csv').as_posix()}', {opts})")
    con.execute("""
    CREATE OR REPLACE TABLE stg.source_to_standard AS
    SELECT src.vocabulary_id AS source_vocabulary_id, src.concept_code AS source_code,
           CAST(src.concept_id AS INTEGER) AS source_concept_id,
           CAST(tgt.concept_id AS INTEGER) AS target_concept_id, tgt.domain_id AS target_domain_id
    FROM stg.concept src
    JOIN stg.concept_relationship cr ON cr.concept_id_1 = src.concept_id AND cr.relationship_id = 'Maps to'
         AND COALESCE(cr.invalid_reason, '') = ''
    JOIN stg.concept tgt ON tgt.concept_id = cr.concept_id_2 AND tgt.standard_concept = 'S'
         AND COALESCE(tgt.invalid_reason, '') = ''
    """)
else:
    con.execute("""
    CREATE OR REPLACE TABLE stg.source_to_standard (
        source_vocabulary_id VARCHAR, source_code VARCHAR, source_concept_id INTEGER,
        target_concept_id INTEGER, target_domain_id VARCHAR)
    """)
print("Vocabularios Athena:", HAS_VOCAB, "| filas de mapeo:", con.execute("SELECT COUNT(*) FROM stg.source_to_standard").fetchall()[0][0])

### Paso 7 — `cdm.condition_occurrence` y `cdm.measurement`

**Explicación línea por línea (SQL)**

- `LEFT JOIN stg.source_to_standard s ON s.source_vocabulary_id = 'SNOMED' AND s.source_code = c.snomed_code AND s.target_domain_id = 'Condition'`: busca el concepto estándar del dominio *Condition*. `LEFT JOIN` conserva la fila aunque no haya mapeo.
- `COALESCE(s.target_concept_id, 0) AS condition_concept_id`: 0 si no hay mapeo.
- `32827 AS condition_type_concept_id`: *EHR encounter record*, el mismo tipo que `ETL-Synthea` usa en visitas, procedimientos y mediciones.
- `c.snomed_code AS condition_source_value`: el código original.
- `LEFT JOIN stg.visit_map v USING (encounter_id)`: enlaza la condición con su visita.
- Para `measurement`: `WHERE o.category IN ('vital-signs', 'laboratory') AND o.value_num IS NOT NULL AND regexp_full_match(o.loinc_code, '[0-9]+-[0-9]')`: solo mediciones numéricas con código de forma LOINC; así quedan fuera `QALY`, `DALY` y similares (hallazgo de la semana 5).
- `o.value_num AS value_as_number`, `o.unit AS unit_source_value`, `o.loinc_code AS measurement_source_value`: valor, unidad y código de origen.
- `0 AS unit_concept_id`: sin vocabulario no se traduce la unidad UCUM; con Athena se mapearía igual que el código.
- `ROW_NUMBER() OVER (ORDER BY ...)`: identificadores enteros deterministas.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE cdm.condition_occurrence AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id, c.start_date, c.snomed_code) AS condition_occurrence_id,
       m.person_id,
       COALESCE(s.target_concept_id, 0) AS condition_concept_id,
       c.start_date                     AS condition_start_date,
       c.stop_date                      AS condition_end_date,
       32827                            AS condition_type_concept_id,
       v.visit_occurrence_id,
       c.snomed_code                    AS condition_source_value,
       COALESCE(s.source_concept_id, 0) AS condition_source_concept_id
FROM silver.conditions c
JOIN stg.person_map m USING (patient_id)
LEFT JOIN stg.visit_map v USING (encounter_id)
LEFT JOIN stg.source_to_standard s
       ON s.source_vocabulary_id = 'SNOMED' AND s.source_code = c.snomed_code AND s.target_domain_id = 'Condition';

CREATE OR REPLACE TABLE cdm.measurement AS
SELECT ROW_NUMBER() OVER (ORDER BY m.person_id, o.obs_ts, o.loinc_code) AS measurement_id,
       m.person_id,
       COALESCE(s.target_concept_id, 0) AS measurement_concept_id,
       o.obs_ts::DATE                   AS measurement_date,
       o.obs_ts                         AS measurement_datetime,
       32827                            AS measurement_type_concept_id,
       o.value_num                      AS value_as_number,
       0                                AS unit_concept_id,
       o.unit                           AS unit_source_value,
       v.visit_occurrence_id,
       o.loinc_code                     AS measurement_source_value,
       COALESCE(s.source_concept_id, 0) AS measurement_source_concept_id
FROM silver.observations o
JOIN stg.person_map m USING (patient_id)
LEFT JOIN stg.visit_map v USING (encounter_id)
LEFT JOIN stg.source_to_standard s
       ON s.source_vocabulary_id = 'LOINC' AND s.source_code = o.loinc_code AND s.target_domain_id = 'Measurement'
WHERE o.category IN ('vital-signs', 'laboratory')
  AND o.value_num IS NOT NULL
  AND regexp_full_match(o.loinc_code, '[0-9]+-[0-9]');
""")
con.sql("""
SELECT 'condition_occurrence' AS tabla, COUNT(*) AS filas FROM cdm.condition_occurrence
UNION ALL SELECT 'measurement', COUNT(*) FROM cdm.measurement
""").df()

### Paso 8 — Chequeos de calidad al estilo del Data Quality Dashboard de OHDSI

El *Data Quality Dashboard* (DQD) de OHDSI ejecuta más de 3.500 chequeos sobre una base OMOP y es requisito antes de participar en estudios de la red OHDSI. Aquí implementas una muestra representativa, organizada con las mismas categorías de Kahn de la semana 2.

**Explicación línea por línea**

- `checks = [...]`: lista de tuplas `(nombre, categoría de Kahn, SQL que devuelve fallos y total)`.
- `person_sin_periodo`: toda persona con datos debe tener un periodo de observación (completitud).
- `condiciones_fuera_periodo`: condiciones fuera del periodo de observación (plausibilidad temporal).
- `visitas_huerfanas`: integridad referencial `visit_occurrence.person_id → person` (conformidad relacional).
- `condition_concept_0` y `measurement_concept_0`: porcentaje de registros sin concepto estándar; sin vocabularios será 100 %, y con Athena debería bajar mucho. **Es el indicador más mirado en un ETL a OMOP.**
- `sbp_implausible`: sistólicas fuera de 50-260 mm[Hg] usando el código de origen.
- `for nombre, cat, sql in checks: fail, total = con.execute(sql).fetchall()[0]`: cada consulta devuelve dos números.
- `pd.DataFrame(filas).to_csv(...)`: guarda el reporte.

In [ ]:
checks = [
    ("person_sin_periodo", "Completeness", """
        SELECT COUNT(*) FILTER (WHERE op.person_id IS NULL), COUNT(*)
        FROM cdm.person p LEFT JOIN cdm.observation_period op USING (person_id)"""),
    ("condiciones_fuera_periodo", "Plausibility", """
        SELECT COUNT(*) FILTER (WHERE c.condition_start_date < op.observation_period_start_date
                                   OR c.condition_start_date > op.observation_period_end_date), COUNT(*)
        FROM cdm.condition_occurrence c JOIN cdm.observation_period op USING (person_id)"""),
    ("visitas_huerfanas", "Conformance", """
        SELECT COUNT(*) FILTER (WHERE p.person_id IS NULL), COUNT(*)
        FROM cdm.visit_occurrence v LEFT JOIN cdm.person p USING (person_id)"""),
    ("condition_concept_0", "Conformance", """
        SELECT COUNT(*) FILTER (WHERE condition_concept_id = 0), COUNT(*) FROM cdm.condition_occurrence"""),
    ("measurement_concept_0", "Conformance", """
        SELECT COUNT(*) FILTER (WHERE measurement_concept_id = 0), COUNT(*) FROM cdm.measurement"""),
    ("sbp_implausible", "Plausibility", """
        SELECT COUNT(*) FILTER (WHERE value_as_number < 50 OR value_as_number > 260), COUNT(*)
        FROM cdm.measurement WHERE measurement_source_value = '8480-6'"""),
]
filas = []
for nombre, cat, sql in checks:
    fail, total = con.execute(sql).fetchall()[0]
    filas.append({"check": nombre, "kahn_category": cat, "n_fail": fail, "n_total": total,
                  "pct_fail": round(100 * fail / total, 2) if total else 0.0})
dq = pd.DataFrame(filas)
dq.to_csv(REPORTS_DIR / "omop_dq.csv", index=False)
dq

### Paso 9 — Una consulta analítica sobre OMOP: tabla de caracterización

La ventaja de OMOP es que **la misma consulta** funciona en cualquier base convertida al CDM en cualquier país.

**Explicación línea por línea (SQL)**

- `WITH htn AS (...)`: personas con hipertensión según el conjunto de conceptos de la semana 5 (`concept_sets.parquet`). Sin vocabularios, filtramos por `condition_source_value`; con Athena se filtraría por `condition_concept_id` y la tabla `CONCEPT_ANCESTOR`.
- `read_parquet('.../concept_sets.parquet') WHERE concept_set = 'hypertension'`: DuckDB lee el Parquet directamente dentro del SQL.
- `last_sbp AS (... QUALIFY ROW_NUMBER() ... = 1)`: última sistólica por persona.
- `2021 - p.year_of_birth AS age_2021`: edad aproximada al año de corte (suficiente para caracterizar).
- `AVG(...)`, `MEDIAN(...)`: estadísticas por grupo; `MEDIAN` es una función agregada nativa de DuckDB.

In [ ]:
cs_path = (DATA_DIR / "reference" / "concept_sets.parquet").as_posix()
con.sql(f"""
WITH htn AS (
    SELECT DISTINCT person_id FROM cdm.condition_occurrence
    WHERE condition_source_value IN (
        SELECT code FROM read_parquet('{cs_path}') WHERE concept_set = 'hypertension')
),
last_sbp AS (
    SELECT person_id, value_as_number AS sbp
    FROM cdm.measurement WHERE measurement_source_value = '8480-6'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_datetime DESC) = 1
)
SELECT CASE WHEN h.person_id IS NULL THEN 'sin HTA' ELSE 'con HTA' END AS grupo,
       COUNT(*)                              AS personas,
       ROUND(AVG(2021 - p.year_of_birth), 1) AS edad_media,
       ROUND(MEDIAN(s.sbp), 1)               AS sbp_mediana
FROM cdm.person p
LEFT JOIN htn h USING (person_id)
LEFT JOIN last_sbp s USING (person_id)
GROUP BY 1
""").df()

### Paso 10 — Publicar el CDM

**Explicación línea por línea**

- `OMOP_TABLES = [...]`: las 6 tablas clínicas construidas.
- `COPY cdm.{t} TO '...' (FORMAT parquet, COMPRESSION zstd)`: misma exportación de la semana 3.
- `con.close()`: cierra.
- **Siguiente nivel (opcional):** el proyecto oficial `OHDSI/dbt-synthea` hace este mismo ETL con dbt sobre DuckDB, con todas las tablas del CDM. Compara tu SQL con el suyo.

In [ ]:
OMOP_TABLES = ["person", "observation_period", "visit_occurrence", "condition_occurrence", "measurement", "death"]
for t in OMOP_TABLES:
    destino = (OMOP_DIR / f"{t}.parquet").as_posix()
    con.execute(f"COPY cdm.{t} TO '{destino}' (FORMAT parquet, COMPRESSION zstd)")
    print(f"cdm.{t:<22} {con.execute(f'SELECT COUNT(*) FROM cdm.{t}').fetchall()[0][0]:>9,} filas")
con.close()

### Autoevaluación

1. ¿Por qué OMOP usa enteros (`person_id`) y además conserva `person_source_value`?
2. ¿Qué significa `condition_concept_id = 0` y por qué no es un error del ETL?
3. ¿Qué pierdes al mapear `wellness`, `ambulatory` y `outpatient` al mismo concepto 9202?
4. Descarga de Athena los vocabularios SNOMED y LOINC, ponlos en `data/vocab/` y vuelve a correr el notebook. ¿Cuánto baja `condition_concept_0`?

**Commit sugerido:** `feat(s06): ETL Synthea → OMOP CDM v5.4 en DuckDB con chequeos estilo DQD`